In [45]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.globals import set_debug
from typing import Literal

from pydantic import BaseModel, Field
from datasets import load_dataset

from data import LABEL_MAP

In [46]:
# Help LLM out with some few shot examples. Currently model tends to predict Business where it is Sci/Tech
# The example_prompt template must match your examples dict keys (text, category, reason).
few_shot_examples = [ # from train dataset
  {
    "text": "Venezuelans Vote Early in Referendum on Chavez Rule (Reuters) Reuters - Venezuelans turned out early\\and in large numbers on Sunday to vote in a historic referendum\\that will either remove left-wing President Hugo Chavez from\\office or give him a new mandate to govern for the next two\\years.",
    "category": "World",
    "reason": "Reports on Venuzuelan elections, protests, politics and civic issues"
    },
 {
    "text": "S.Koreans Clash with Police on Iraq Troop Dispatch (Reuters) Reuters - South Korean police used water cannon in\\central Seoul Sunday to disperse at least 7,000 protesters\\urging the government to reverse a controversial decision to\\send more troops to Iraq.",
    "category": "World",
    "reason": "Focuses on protests in South Korea in relation to a government decision to participate in conflict in Iraq"},
 {
    "text": "Phelps, Thorpe Advance in 200 Freestyle (AP) AP - Michael Phelps took care of qualifying for the Olympic 200-meter freestyle semifinals Sunday, and then found out he had been added to the American team for the evening's 400 freestyle relay final. Phelps' rivals Ian Thorpe and Pieter van den Hoogenband and teammate Klete Keller were faster than the teenager in the 200 free preliminaries.",
    "category": "Sports",
    "reason": "Covers swimming competition results"
    },
 {
    "text": "Reds Knock Padres Out of Wild-Card Lead (AP) AP - Wily Mo Pena homered twice and drove in four runs, helping the Cincinnati Reds beat the San Diego Padres 11-5 on Saturday night. San Diego was knocked out of a share of the NL wild-card lead with the loss and Chicago's victory over Los Angeles earlier in the day.",
    "category": "Sports",
    "reason": "Describes results from a national baseball league game"
    },
 {
    "text": "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
    "category": "Business",
    "reason": "Describes stock market movements on wall street stock exchange"
    },
 {
    "text": "Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.",
     "category": "Business",
     "reason": "Discusses a private investment firm's investment in an aerospace company"
     },
 {
    "text": "Madden, ESPN Football Score in Different Ways (Reuters) Reuters - Was absenteeism a little high\\on Tuesday among the guys at the office? EA Sports would like\\to think it was because 'Madden NFL 2005' came out that day,\\and some fans of the football simulation are rabid enough to\\take a sick day to play it.",
    "category": "Sci/Tech",
    "reason": "Describes potential workpolace absenteeism due to a new video game, Madden NFL 2005, released by video game company EA Sports"
    },
 {
    "text": "Group to Propose New High-Speed Wireless Format (Reuters) Reuters - A group of technology companies\\including Texas Instruments Inc. (TXN.N), STMicroelectronics\\(STM.PA) and Broadcom Corp. (BRCM.O), on Thursday said they\\will propose a new wireless networking standard up to 10 times\\the speed of the current generation.",
    "category": "Sci/Tech",
    "reason": "Focuses on a group of technology companies proposing a new wireless networking standard"
    },
  ]
few_shot_example_prompt = ChatPromptTemplate.from_messages([
      ("human", "{text}"),
      ("ai", "{category} — {reason}"),
  ])                                                                                                                  
  
few_shot_prompt = FewShotChatMessagePromptTemplate(                                                                 
      examples=few_shot_examples,
      example_prompt=few_shot_example_prompt,
  ) 

In [55]:
# Define prompt template
classification_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an experienced newspaper editor who has experience working with content across news categories. "
        "Classify the article into exactly one of: World, Sports, Business, Sci/Tech. "
        "Respond with the category and a brief one-sentence reason."
    )),
    few_shot_prompt,
    ("human", "{text}"),
])


ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an experienced newspaper editor who has experience working with content across news categories. Classify the article into exactly one of: World, Sports, Business, Sci/Tech. Respond with the category and a brief one-sentence reason.'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'text': 'Venezuelans Vote Early in Referendum on Chavez Rule (Reuters) Reuters - Venezuelans turned out early\\and in large numbers on Sunday to vote in a historic referendum\\that will either remove left-wing President Hugo Chavez from\\office or give him a new mandate to govern for the next two\\years.', 'category': 'World', 'reason': 'Reports on Venuzuelan elections, protests, politics and civic issues'}, {'text': 'S.Koreans Clash with Police on Iraq Troop Dispatch (Reute

In [48]:
# Define response schema
class ClassificationResult(BaseModel):
    category: Literal["World", "Sports", "Business", "Sci/Tech"] = Field(
        description="The news category that best describes the content. Valid values are World, Sports, Business, Sci/Tech"
    )
    reasoning: str = Field(
        description="A brief one-sentence explanation of why this category was chosen"
    )


In [49]:
# Define model
model_name = "qwen2.5:3b"

llm = ChatOllama(model=model_name)
structured_llm = llm.with_structured_output(ClassificationResult)

In [50]:
# Create chain (RunnableSequence)
chain = classification_prompt | structured_llm

### Load data

In [51]:
def load_sample(split: str = "test", n: int = 100):
    ds = load_dataset("fancyzhx/ag_news")
    return ds[split].select(range(n))

sample_data = load_sample("test", 10)

In [52]:
def load_sample(split: str = "test", n: int = 100):
    ds = load_dataset("fancyzhx/ag_news")
    return ds[split].select(range(n))

In [53]:
def load_few_shot_examples(n_per_class: int = 2) -> list[dict]:
    ds = load_dataset("fancyzhx/ag_news")
    train = ds["train"]
    examples = []
    for label_id, label_name in LABEL_MAP.items():
        class_rows = train.filter(lambda x: x["label"] == label_id).select(range(n_per_class))
        for row in class_rows:
            examples.append({"text": row["text"], "category": label_name})
    return examples

load_few_shot_examples(2)

[{'text': 'Venezuelans Vote Early in Referendum on Chavez Rule (Reuters) Reuters - Venezuelans turned out early\\and in large numbers on Sunday to vote in a historic referendum\\that will either remove left-wing President Hugo Chavez from\\office or give him a new mandate to govern for the next two\\years.',
  'category': 'World'},
 {'text': 'S.Koreans Clash with Police on Iraq Troop Dispatch (Reuters) Reuters - South Korean police used water cannon in\\central Seoul Sunday to disperse at least 7,000 protesters\\urging the government to reverse a controversial decision to\\send more troops to Iraq.',
  'category': 'World'},
 {'text': "Phelps, Thorpe Advance in 200 Freestyle (AP) AP - Michael Phelps took care of qualifying for the Olympic 200-meter freestyle semifinals Sunday, and then found out he had been added to the American team for the evening's 400 freestyle relay final. Phelps' rivals Ian Thorpe and Pieter van den Hoogenband and teammate Klete Keller were faster than the teenage

### Generate response
- with debug

In [54]:
set_debug(True)
response = chain.invoke({"text": sample_data[0]["text"]})
set_debug(False)

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "text": "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul."
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "text": "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul."
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "System: You are an experienced newspaper editor who has experience working with content across news categories. Classify the article into exactly one of: World, Sports, Business, Sci/Tech. Respond with the category and a brief one-sentence reason.\nHuman: Vene